# Eksperimen RQ1 (3 model) + RQ2 - vLLM

Notebook lengkap: **install + token + clone + run + push**, semua di sini. Tinggal isi token di
cell 2 lalu **Run All**.

- GPU besar (A6000 48 GB) -> Qwen2.5-7B presisi penuh, otomatis.
- RQ1 lintas 3 model: Qwen2.5-7B, Llama-3.1-8B, Gemma-2-9B (tiap model proses sendiri -> tak OOM).
- Config GPU dipilih otomatis di kode (`experiments/_kaggle_env.py`).

> **Token nempel di file .ipynb ini setelah diisi.** Jangan commit / share notebook-nya.

## 1. Install library

In [ ]:
!pip install -q vllm numpy huggingface_hub

## 2. Isi token
- **HF_TOKEN**: download Llama & Gemma (gated). Terima lisensi dulu di halaman HF model, lalu Settings -> Access Tokens.
- **GITHUB_TOKEN**: push hasil ke repo (opsional). Set `''` kalau mau unduh hasil manual.

In [ ]:
import os
HF_TOKEN     = 'hf_xxx'    # <-- GANTI token HF-mu
GITHUB_TOKEN = 'ghp_xxx'   # <-- GANTI token GitHub-mu (atau '' untuk skip push)

if HF_TOKEN and not HF_TOKEN.startswith('hf_x'):
    os.environ['HF_TOKEN'] = HF_TOKEN
if GITHUB_TOKEN and not GITHUB_TOKEN.startswith('ghp_x'):
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
print('HF_TOKEN diset    :', 'HF_TOKEN' in os.environ)
print('GITHUB_TOKEN diset:', 'GITHUB_TOKEN' in os.environ)

## 3. Ambil repo
Pakai repo yang ada kalau notebook dibuka di dalamnya (tarik update); kalau berdiri sendiri, clone fresh ke `~/creditaudit`.

In [ ]:
import os, subprocess, torch
from pathlib import Path

def _find_repo():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'gearts' / 'harness.py').exists():
            return cand
    return None

root = _find_repo()
if root is None:
    root = Path.home() / 'creditaudit'
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/eycoo/creditaudit.git', str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull'], check=False)
else:
    subprocess.run(['git', '-C', str(root), 'pull'], check=False)

os.chdir(root)
print('REPO_ROOT =', root, '| GPU:', torch.cuda.device_count())
assert os.path.isfile('src/gearts/harness.py'), 'repo tidak lengkap'
data = 'data/processed/benchmark_acuan.jsonl'
assert os.path.isfile(data), f'{data} tidak ada'
print('benchmark OK:', sum(1 for _ in open(data, encoding='utf-8')), 'baris')

## 4. DIAGNOSTIK - output MENTAH model (4 sampel)
Cek model ikut format `LANGKAH`/`JAWABAN` dan label benar.

In [ ]:
!python experiments/debug_dump.py 4

## 5. RQ1 - 3 model
Qwen2.5-7B, Llama-3.1-8B, Gemma-2-9B; tiap model proses terpisah. Tanpa HF_TOKEN, Llama/Gemma dilewati (Qwen tetap jalan).

In [ ]:
!python experiments/run_rq1_multi.py

## 6. RQ2 - kurva grounding vs token (model anchor)

In [ ]:
!python experiments/exp2_rq2.py

## 7. Lihat hasil

In [ ]:
for p in ('experiments/rq1/tabel_rq1.md', 'experiments/rq2/kurva_rq2.md'):
    print('=' * 8, p, '=' * 8)
    print(open(p, encoding='utf-8').read())

## 8. Push hasil ke repo (butuh GITHUB_TOKEN)
Kalau GITHUB_TOKEN kosong -> skip; unduh manual `experiments/rq1` & `rq2`. Setelah push, di laptop jalankan `git pull`.

In [ ]:
import os
if os.environ.get('GITHUB_TOKEN'):
    get_ipython().system('bash scripts/push_results.sh')
else:
    print('GITHUB_TOKEN kosong -> skip push. Unduh manual experiments/rq1 & experiments/rq2.')